## Loading data from yfinance

Ticker GSPC is same as the S&P 500 index. It is the specific ticker symbol used by Yahoo Finance to represent the Standard & Poor's 500 index, which tracks 500 large-cap U.S. We will use the daily adjusted close price of this ticker from 2015 to 2025.

In [33]:
!pip install yfinance --upgrade -q

In [34]:
# import yfinance as yf
# import pandas as pd

# data = yf.download('^GSPC', start='2015-01-01', end='2024-12-31')

In [35]:
import numpy as np
import pandas as pd
import yfinance as yf

TICKER = '^GSPC'
START_DATE = '2014-12-03' # including 20 days buffer period so that we have all days of 2015 after votility calculation
END_DATE = '2025-12-31'
TRAIN_END = '2020-12-31'
TEST_START = '2021-01-01'

# Download raw OHLCV data.
# auto_adjust=False keeps both Close and Adj Close when available.
# We generally want to use Adj Close that ignore stock split and dividend for return over time
# Using Close would introduce fake volatility and incorrect returns
raw = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)

if raw.empty:
    raise ValueError('No data was downloaded. Check ticker symbol, dates, or internet connection.')


---
## Data Cleaning

In [36]:
# Flatten columns in case yfinance returns a MultiIndex.
# This is a small defensive fix for a quirk in how yfinance sometimes formats its data.
# Usually, yfinance returns a DataFrame with raw.columns is a simple list (Index)
# Sometimes (especially with multiple tickers or certain settings), we get MultiIndex columns
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

# SAnity check to make sure the raw data has all the required columns
required_cols = {'Adj Close', 'Volume'}
missing = required_cols.difference(raw.columns)
if missing:
    raise ValueError(f'Missing expected columns: {missing}. Available columns: {list(raw.columns)}')

# Create a clean dataframe to store only the info that we care about
data = pd.DataFrame(index=raw.index)
data['Adj_Close'] = raw['Adj Close']
data['Volume'] = raw['Volume']
data['Log_Return'] = np.log(data['Adj_Close'] / data['Adj_Close'].shift(1))  # daily log return, log return = log(P_t/P_(t-1)), more stable for modeling, especially Gaussian assumptions in HMM
data['Volatility'] = data['Log_Return'].rolling(window=20).std()  # Calculates standard deviation of returns over the last 20 days

# Keep only rows that does not have NaN (the first 20 rows will have NaN because we don't have enough data points)
data = data.dropna().copy()

---
## Data set split
- Training set: 2015-01-01 to 2020-12-31
- Test set: 2021-01-01 to 2025-12-31

In [37]:
# from numpy import testing

# # daily returns
# data['Return'] = data['Close'].pct_change()

# # volatility
# data['Volatility'] = data['Return'].rolling(window=20).std()

# # split data
# training_data = data[data.index <= '2019-12-31']
# testing_data = data[data.index >= '2020-01-01']


# tests
# print(f"Tracking {data.shape[0]} days over {data.shape[1]} categories")
# print(f"Date range: {data.index[0]} to {data.index[-1]}")
# print(data[['Close', 'Return', 'Volatility']].head())


In [38]:
# Split data into train and test based on the date
training_data = data.loc[:TRAIN_END].copy()
testing_data = data.loc[TEST_START:].copy()

if training_data.empty or testing_data.empty:
    raise ValueError(
        f'Unexpected empty split. Training rows: {len(training_data)}, testing rows: {len(testing_data)}'
    )

---
## Standardize input features using training-set statistics (z-score scaling)

This step rescales each feature to have mean ≈ 0 and standard deviation ≈ 1

     z = (x - μ_train) / σ_train

- Statistics (mean and std) are computed ONLY from the training data to prevent data leakage from the test set.
- The same training statistics are then applied to both training and testing datasets to ensure consistency.
- Standardization is important because features (e.g., volume vs returns) have very different scales. Without scaling, large-magnitude features would dominate the model.
- Any zero standard deviations are replaced with NaN to avoid division errors.

In [39]:
# Standardize model inputs using training statistics only.
feature_cols = ['Log_Return', 'Volume', 'Volatility']
train_means = training_data[feature_cols].mean()
train_stds = training_data[feature_cols].std().replace(0, np.nan)

training_data_std = training_data.copy()
testing_data_std = testing_data.copy()
training_data_std[feature_cols] = (training_data[feature_cols] - train_means) / train_stds
testing_data_std[feature_cols] = (testing_data[feature_cols] - train_means) / train_stds

# Drop rows with any NaNs created during scaling.
training_data_std = training_data_std.dropna().copy()
testing_data_std = testing_data_std.dropna().copy()

print(f'Ticker: {TICKER}')
print(f'Full sample: {data.index.min().date()} to {data.index.max().date()} ({len(data)} rows)')
print(f'Training: {training_data_std.index.min().date()} to {training_data_std.index.max().date()} ({len(training_data_std)} rows)')
print(f'Testing:  {testing_data_std.index.min().date()} to {testing_data_std.index.max().date()} ({len(testing_data_std)} rows)')
print('Columns in exported datasets:')
print(list(training_data_std.columns))

training_data_std.head()

Ticker: ^GSPC
Full sample: 2015-01-02 to 2025-12-30 (2765 rows)
Training: 2015-01-02 to 2020-12-31 (1511 rows)
Testing:  2021-01-04 to 2025-12-30 (1254 rows)
Columns in exported datasets:
['Adj_Close', 'Volume', 'Log_Return', 'Volatility']


,Adj_Close,Volume,Log_Return,Volatility
Date,,,,
2015-01-02,2058.199951,-1.193727,-0.062474,0.100186
2015-01-05,2020.579956,-0.053625,-1.595492,0.204821
2015-01-06,2002.609985,0.637481,-0.790007,0.219297
2015-01-07,2025.900024,-0.046975,0.945255,0.260521
2015-01-08,2062.139893,0.087411,1.467412,0.359755


---
## Save train and test data to csv files

In [40]:
training_data_std.to_csv('sp500_train.csv')
testing_data_std.to_csv('sp500_test.csv')

print(f"Training: {training_data.shape[0]} days")
print(f"Testing: {testing_data.shape[0]} days")

Training: 1511 days
Testing: 1254 days
